In [ ]:
# last_verified: 2026-09-18 · Checkov n/a

# Checkov — Cross-Module Scanning Limitations: Static Directory vs Plan JSON

Comparison of Checkov's two primary invocation modes for cross-module IaC scanning.

## Purpose

Examine the limitations of Checkov's cross-module scanning when comparing the two primary invocation modes: a static directory scan against a Terraform plan JSON scan. Understanding these limitations helps determine which approach is appropriate for a given infrastructure codebase.

## When to Use

- **Static directory scan** — When you need to scan raw `.tf` files directly without executing Terraform. Useful for quick policy enforcement and CI gates.
- **Plan JSON scan** — When you need to scan the resolved state of your infrastructure after Terraform has computed the execution plan. Useful for catching issues that only manifest after Terraform resolves references across modules.

## Prerequisites

- A Terraform project with multiple modules
- Checkov installed
- `terraform` CLI available

## Step 1: Create a sample multi-module Terraform project

The sample project has a root module that calls a child module. This structure exposes how cross-module references are handled differently depending on the scan mode.

In [ ]:
mkdir -p /tmp/checkov-demo/modules/vpc\ncd /tmp/checkov-demo\ncat > main.tf << 'EOF'\nmodule "vpc" {\n  source = "./modules/vpc"\n  cidr_block = "10.0.0.0/16"\n}\nEOF\ncat > modules/vpc/main.tf << 'EOF'\nresource "aws_vpc" "main" {\n  cidr_block           = var.cidr_block\n  enable_dns_support   = true\n  enable_dns_hostnames = true\n}\n\nvariable "cidr_block" {\n  type    = string\n  default = "10.0.0.0/16"\n}\nEOF\n

## Step 2: Static directory scan

Running Checkov against the directory parses raw `.tf` files. Cross-module references are evaluated at the file level, which means variables and outputs defined in child modules may not be fully resolved.

In [ ]:
checkov -d /tmp/checkov-demo --framework terraform


## Step 3: Terraform plan JSON scan

Running `terraform plan -out=tfplan` followed by `terraform show -json tfplan` and piping the result to Checkov provides a fully resolved view. Cross-module references are resolved by Terraform before Checkov evaluates them.

In [ ]:
cd /tmp/checkov-demo\nterraform init -backend=false\nterraform plan -out=tfplan\nterraform show -json tfplan | checkov -f - --framework terraform


## Step 4: Compare results and identify limitations

The static directory scan may miss findings that depend on resolved cross-module references. For example:
- Variables passed between modules may not be validated until the plan JSON is generated.
- Outputs from child modules referenced in the root module are only available after Terraform resolves the plan.
- The plan JSON scan captures the full dependency graph, but the static scan sees files in isolation.

Conversely, the plan JSON scan requires Terraform to be installed and configured, and it cannot be run without a valid backend configuration. The static scan is faster and works without any Terraform execution.

In [ ]:
echo "Static scan findings:"
checkov -d /tmp/checkov-demo --framework terraform --quiet | head -20
echo ""
echo "Plan JSON scan findings:"
cd /tmp/checkov-demo && terraform show -json tfplan | checkov -f - --framework terraform --quiet | head -20


## Summary

Checkov's cross-module scanning has inherent limitations that differ between invocation modes. The static directory scan is fast and requires no Terraform execution but may miss findings that depend on resolved cross-module references. The plan JSON scan provides a complete picture of the infrastructure after Terraform has resolved all modules, but requires a full Terraform setup. For comprehensive IaC security scanning, the plan JSON approach is recommended when module cross-references are material to the security posture.